In [1]:
import os
import sys
import shutil
import numpy as np
from osgeo import gdal
import subprocess

def multilook_and_extract(slc_vrt, cor_file, output_tif, range_looks=4, az_looks=1):
    if not os.path.exists(slc_vrt):
        print(f"  [!] Error: {slc_vrt} not found.")
        return

    print(f"\n--- Extracting Amplitude from {slc_vrt} ---")
    
    # 1. Read the Coherence file to get the EXACT target dimensions
    ds_cor = gdal.Open(cor_file, gdal.GA_ReadOnly)
    target_w = ds_cor.RasterXSize
    target_h = ds_cor.RasterYSize
    ds_cor = None

    # 2. Read complex SLC data
    print("  > Reading full resolution SLC...")
    ds_slc = gdal.Open(slc_vrt, gdal.GA_ReadOnly)
    comp_data = ds_slc.GetRasterBand(1).ReadAsArray()
    ds_slc = None

    # 3. Calculate Amplitude
    print("  > Calculating Amplitude...")
    amp_data = np.abs(comp_data).astype(np.float32)

    # 4. Multilook the Amplitude (Block Average matching your topsApp.xml)
    print(f"  > Multilooking (Range looks: {range_looks}, Azimuth looks: {az_looks})...")
    h, w = amp_data.shape
    h_crop = (h // az_looks) * az_looks
    w_crop = (w // range_looks) * range_looks
    
    amp_cropped = amp_data[:h_crop, :w_crop]
    amp_ml = amp_cropped.reshape(h // az_looks, az_looks, w // range_looks, range_looks).mean(axis=(1, 3))

    # 5. Align to the Coherence grid
    print("  > Aligning to Coherence grid...")
    final_amp = np.zeros((target_h, target_w), dtype=np.float32)
    min_h = min(target_h, amp_ml.shape[0])
    min_w = min(target_w, amp_ml.shape[1])
    final_amp[:min_h, :min_w] = amp_ml[:min_h, :min_w]

    # 6. Disguise!
    print("  > Injecting into ISCE pipeline...")
    cor_bak = cor_file + ".bak"
    shutil.copy(cor_file, cor_bak) # Backup the original

    # Use GDAL Update mode so we don't break the BIL/BIP band interleaving
    ds_cor_update = gdal.Open(cor_file, gdal.GA_Update)
    ds_cor_update.GetRasterBand(1).WriteArray(final_amp)
    ds_cor_update.GetRasterBand(2).WriteArray(np.zeros((target_h, target_w), dtype=np.float32))
    ds_cor_update.FlushCache()
    ds_cor_update = None

    # 7. Run ISCE Geocode
    isce_apps_script = '/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/applications/topsApp.py'

    # 2. Use the active kernel's Python executable to run it
    # This forces the OS to use env39's Python to run env39's topsApp.py
    print(f"Forcing execution using Python interpreter: {sys.executable}")
    print(f"Running script: {isce_apps_script}")

    # Instead of passing the script alone, we pass [python_interpreter, script_path, flags]
    subprocess.run([sys.executable, isce_apps_script, "--dostep=geocode"])

    # 8. Save the geocoded result
    print("  > Saving final GeoTIFF...")
    geo_file = cor_file + ".geo"
    ds_geo = gdal.Open(geo_file)
    driver = gdal.GetDriverByName("GTiff")
    
    # We only want Band 1 (our amplitude)
    out_ds = driver.Create(output_tif, ds_geo.RasterXSize, ds_geo.RasterYSize, 1, gdal.GDT_Float32)
    out_ds.SetGeoTransform(ds_geo.GetGeoTransform())
    out_ds.SetProjection(ds_geo.GetProjection())
    out_ds.GetRasterBand(1).WriteArray(ds_geo.GetRasterBand(1).ReadAsArray())
    out_ds.FlushCache()
    out_ds = None
    ds_geo = None

    # 9. Restore the original coherence file
    shutil.move(cor_bak, cor_file)
    print(f"✅ Success! Saved {output_tif}")

if __name__ == "__main__":
    ref_vrt = "merged/reference.slc.full.vrt"
    sec_vrt = "merged/secondary.slc.full.vrt"
    cor_file = "merged/topophase.cor"

    multilook_and_extract(ref_vrt, cor_file, "reference_amplitude.tif")
    multilook_and_extract(sec_vrt, cor_file, "secondary_amplitude.tif")

    print("\nRestoring original geographic coherence file...")
    subprocess.run(["topsApp.py", "--dostep=geocode"], stdout=subprocess.DEVNULL)
    print("All done! Perfect pixel-to-pixel match achieved.")


--- Extracting Amplitude from merged/reference.slc.full.vrt ---
  > Reading full resolution SLC...
  > Calculating Amplitude...
  > Multilooking (Range looks: 4, Azimuth looks: 1)...
  > Aligning to Coherence grid...
  > Injecting into ISCE pipeline...
Forcing execution using Python interpreter: /home/st-juho/code_testing/miniconda3/envs/env39/bin/python
Running script: /home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/applications/topsApp.py
2026-06-09 15:59:35,909 - isce.insar - INFO - ISCE VERSION = 2.6.3, RELEASE_SVN_REVISION = ,RELEASE_DATE = 20230418, CURRENT_SVN_REVISION = 
ISCE VERSION = 2.6.3, RELEASE_SVN_REVISION = ,RELEASE_DATE = 20230418, CURRENT_SVN_REVISION = 
Step processing
Cannot open PICKLE/unwrap2stage
Running step geocode
2026-06-09 15:59:36,133 - isce.topsinsar.runGeocode - INFO - Geocoding Image
Number of products to geocode:  5
  > Saving final GeoTIFF...


Traceback (most recent call last):
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/applications/topsApp.py", line 1077, in <module>
    insar.run()
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/iscesys/Component/Application.py", line 142, in run
    exitStatus = self._processSteps()
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/iscesys/Component/Application.py", line 405, in _processSteps
    result = func(*pargs, **kwargs)
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/isceobj/TopsProc/Factories.py", line 40, in __call__
    return self.method(self.other, *args, **kwargs)
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/isceobj/TopsProc/runGeocode.py", line 56, in runGeocode
    swathList = self._insar.getValidSwathList(self.swaths

✅ Success! Saved reference_amplitude.tif

--- Extracting Amplitude from merged/secondary.slc.full.vrt ---
  > Reading full resolution SLC...
  > Calculating Amplitude...
  > Multilooking (Range looks: 4, Azimuth looks: 1)...
  > Aligning to Coherence grid...
  > Injecting into ISCE pipeline...
Forcing execution using Python interpreter: /home/st-juho/code_testing/miniconda3/envs/env39/bin/python
Running script: /home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/applications/topsApp.py
2026-06-09 16:00:03,164 - isce.insar - INFO - ISCE VERSION = 2.6.3, RELEASE_SVN_REVISION = ,RELEASE_DATE = 20230418, CURRENT_SVN_REVISION = 
ISCE VERSION = 2.6.3, RELEASE_SVN_REVISION = ,RELEASE_DATE = 20230418, CURRENT_SVN_REVISION = 
Step processing
Cannot open PICKLE/unwrap2stage
Running step geocode
2026-06-09 16:00:03,201 - isce.topsinsar.runGeocode - INFO - Geocoding Image
Number of products to geocode:  5
  > Saving final GeoTIFF...


Traceback (most recent call last):
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/applications/topsApp.py", line 1077, in <module>
    insar.run()
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/iscesys/Component/Application.py", line 142, in run
    exitStatus = self._processSteps()
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/iscesys/Component/Application.py", line 405, in _processSteps
    result = func(*pargs, **kwargs)
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/isceobj/TopsProc/Factories.py", line 40, in __call__
    return self.method(self.other, *args, **kwargs)
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/isceobj/TopsProc/runGeocode.py", line 56, in runGeocode
    swathList = self._insar.getValidSwathList(self.swaths

✅ Success! Saved secondary_amplitude.tif

Restoring original geographic coherence file...
All done! Perfect pixel-to-pixel match achieved.


Traceback (most recent call last):
  File "/home/st-juho/code_testing/miniconda3/envs/env312/lib/python3.12/site-packages/isce/applications/topsApp.py", line 1077, in <module>
    insar.run()
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/iscesys/Component/Application.py", line 142, in run
    exitStatus = self._processSteps()
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/iscesys/Component/Application.py", line 405, in _processSteps
    result = func(*pargs, **kwargs)
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/isceobj/TopsProc/Factories.py", line 40, in __call__
    return self.method(self.other, *args, **kwargs)
  File "/home/st-juho/code_testing/miniconda3/envs/env39/lib/python3.9/site-packages/isce/components/isceobj/TopsProc/runGeocode.py", line 56, in runGeocode
    swathList = self._insar.getValidSwathList(self.swat